# QA thresholds from the validation visits

Suggested `WARN`/`FAIL` thresholds for the `imageQualityQa` metrics, derived from the
validation visit set by the procedure in [`qa-principles.md`](qa-principles.md#deriving-thresholds),
and plotted against the known-good and known-bad data.

The inputs are the stored `iqQaMetrics` of the validation visits: run `imageQualityQa` over
them first (current code, one output collection; never a collection of mixed code versions).
The first run reads them from the Butler, read-only, and caches them as parquet; later runs
read the cache. Delete it to re-read.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pfs.drp.qa.metrics.calibration import DEFAULT_METRICS, METRIC_SPECS, calibrate, labelRows
from pfs.drp.qa.metrics.readers import readMetrics
from pfs.drp.qa.metrics.validationVisits import loadValidationVisits
from pfs.drp.qa.plotting import plotThresholds

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

## Parameters

In [ ]:
BUTLER = "/work/datastore"
COLLECTIONS = ["u/wtgee/tickets/PIPE2D-1914/validation"]
CACHE = (
    Path(os.environ.get("DRP_QA_CACHE", "~/.cache/drp_qa")).expanduser() / "iqQaMetrics-validation.parquet"
)
METRICS = [*DEFAULT_METRICS, "nLines"]

## Read the metrics

In [ ]:
visitSet = loadValidationVisits()
print(
    f"{len(visitSet.knownGood)} known_good entries ({len(visitSet.goodVisits)} visits), "
    f"{len(visitSet.confirmedBad)} known_bad, {len(visitSet.unconfirmedBad)} unconfirmed"
)

if CACHE.exists():
    metrics = pd.read_parquet(CACHE)
else:
    from lsst.daf.butler import Butler

    butler = Butler(BUTLER, collections=COLLECTIONS, writeable=False)
    metrics = readMetrics(butler, visitSet.visits, collections=COLLECTIONS)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    metrics.to_parquet(CACHE)
print(f"{len(metrics)} rows over {metrics['visit'].nunique()} visits from {CACHE}")
missing = sorted(set(visitSet.visits) - set(metrics["visit"]))
print(f"{len(missing)} validation visits have no metrics: {missing}")

## Suggested thresholds

One row per metric and population. Columns to read before trusting a row:
`nGood`/`nGoodVisits` (sample size), `failBounded` (whether the sample bounds the FAIL
percentile), `goodFlaggedWarn`/`goodFlaggedFail` (what the thresholds flag of the good
data), `degenerate`, and `badOk` (whether the known-bad rows reach their expected verdict;
empty when none name the metric in that population).

In [ ]:
table = calibrate(metrics, visitSet, METRICS)
columns = [
    "metric",
    "group",
    "nGood",
    "nGoodVisits",
    "median",
    "robustRms",
    "warn",
    "fail",
    "failLow",
    "failHigh",
    "goodFlaggedWarn",
    "goodFlaggedFail",
    "reliable",
    "failBounded",
    "degenerate",
    "nBad",
    "badOk",
    "nUnconfirmed",
    "unconfirmedFlagged",
]
table[columns].round(4)

## Known-bad checks

Every population with known-bad rows, and what they reached.

In [ ]:
checked = table[table["nBad"] > 0]
for _, row in checked.iterrows():
    print(f"{row['metric']}[{row['group']}]: {row['badSummary']}")

## Plots

Per population: the CDF of the known-good values, WARN (amber) and FAIL (red) with the
95 % interval on the FAIL percentile shaded, and below it the known-bad values (crosses
expect FAIL, triangles WARN) and the unconfirmed ones (open circles). A known-bad marker
left of FAIL (right of it, for `nLines`) is a fault the threshold misses.

In [ ]:
for metric in METRICS:
    labelled = labelRows(metrics, visitSet, metric)
    plotThresholds(labelled, table, metric, absolute=METRIC_SPECS[metric].absolute)
    plt.show()

## Provenance

The sentences for the config fields' `doc`, for the suggestions you adopt.

In [ ]:
for _, row in table[table["nGood"] > 0].iterrows():
    print(f"{row['metric']}[{row['group']}]: warn={row['warn']:g} fail={row['fail']:g}")
    print(f"    {row['provenance']}")